# 02 · Preprocesamiento — Capology

Notebook de preprocesamiento de los datos salariales crudos obtenidos de **Capology**.
El objetivo es limpiar la estructura irregular que produce ScraperFC y estandarizar
todas las temporadas a un conjunto común de columnas.

**Acciones realizadas:**
1. Eliminación de la fila 0 (header real que ScraperFC deja como dato)
2. Renombrado de columnas a nombres descriptivos y limpios
3. Selección de columnas comunes a todas las temporadas
4. Parseo de valores monetarios (de string a float)
5. Corrección de tipos de datos
6. Reordenación de columnas

**Estructura de archivos:**
- Entrada:  `data/raw/capology/cg_<liga>_<temporada>.csv` (36 archivos)
- Salida:   `data/processed/capology/cg_<liga>_<temporada>.csv` (36 archivos)

---

## 1. Imports y configuración de rutas

In [29]:
import pandas as pd
from pathlib import Path

# Ruta raíz del proyecto (dos niveles arriba desde notebooks/02_preprocesamiento/)
ROOT = Path.cwd().parents[1]

RAW_DIR       = ROOT / 'data' / 'raw' / 'capology'
PROCESSED_DIR = ROOT / 'data' / 'processed' / 'capology'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Rutas configuradas')
print(f'   Root:      {ROOT}')
print(f'   Raw:       {RAW_DIR}')
print(f'   Processed: {PROCESSED_DIR}')

✅ Rutas configuradas
   Root:      d:\USER\Desktop\TFM
   Raw:       d:\USER\Desktop\TFM\data\raw\capology
   Processed: d:\USER\Desktop\TFM\data\processed\capology


---
## 2. Prototipo sobre `cg_spain_2425.csv`

Desarrollamos y validamos la lógica de preprocesamiento sobre un único archivo piloto.

**Particularidad de Capology:** ScraperFC devuelve los datos con la fila de cabecera
incluida como primera fila de datos, y con nombres de columna genéricos heredados del HTML
(`Unnamed: 0`, `EST. BASE SALARY`, `BIO`...). Ambos problemas se corrigen aquí.

### 2.1 Carga e inspección inicial

In [30]:
df_proto = pd.read_csv(RAW_DIR / 'cg_spain_2425.csv')

print(f'Dimensiones: {df_proto.shape[0]} filas x {df_proto.shape[1]} columnas')
print()
print('Fila 0 (header real embebido como dato):')
print(df_proto.iloc[0].to_string())
print()
df_proto.head(5)

Dimensiones: 552 filas x 10 columnas

Fila 0 (header real embebido como dato):
Unnamed: 0                       PLAYER
EST. BASE SALARY       GROSS P/W\n(EUR)
EST. BASE SALARY.1     GROSS P/Y\n(EUR)
EST. BASE SALARY.2    ADJ. GROSS\n(EUR)
BIO                                POS.
BIO.1                               AGE
BIO.2                           COUNTRY
Unnamed: 7                         CLUB
data_country                        NaN
data_season                         NaN



,Unnamed: 0,EST. BASE SALARY,EST. BASE SALARY.1,EST. BASE SALARY.2,BIO,BIO.1,BIO.2,Unnamed: 7,data_country,data_season
0,PLAYER,GROSS P/W\n(EUR),GROSS P/Y\n(EUR),ADJ. GROSS\n(EUR),POS.,AGE,COUNTRY,CLUB,NaN,NaN
1,Robert Lewandowski,"€ 640,962","€ 33,330,000","€ 33,330,000",F,36,Poland,Barcelona,spain,2425.0
2,Kylian Mbappé,"€ 600,962","€ 31,250,000","€ 31,250,000",F,26,France,Real Madrid,spain,2425.0
3,Vinicius Junior,"€ 480,769","€ 25,000,000","€ 25,000,000",F,24,Brazil,Real Madrid,spain,2425.0
4,David Alaba,"€ 432,692","€ 22,500,000","€ 22,500,000",D,32,Austria,Real Madrid,spain,2425.0


### 2.2 Eliminación de la fila 0 y renombrado de columnas

La fila 0 contiene los nombres reales de las columnas (`PLAYER`, `GROSS P/W`, etc.)
que ScraperFC no interpreta correctamente. Se elimina y se renombran las columnas
manualmente a nombres limpios en snake_case.

In [31]:
# Eliminar fila 0
df_proto = df_proto.iloc[1:].reset_index(drop=True)

# Renombrar columnas al subconjunto estándar común a todas las temporadas
# (las columnas extra de contrato presentes solo en 25/26 se descartan)
RENAME_MAP = {
    'Unnamed: 0'        : 'player',
    'EST. BASE SALARY'  : 'gross_weekly_eur',
    'EST. BASE SALARY.1': 'gross_annual_eur',
    'BIO'               : 'position',
    'BIO.1'             : 'age',
    'BIO.2'             : 'nationality',
    'Unnamed: 7'        : 'club'
}
df_proto = df_proto.rename(columns=RENAME_MAP)

print('✅ Fila 0 eliminada y columnas renombradas')
print(f'   Dimensiones: {df_proto.shape}')

✅ Fila 0 eliminada y columnas renombradas
   Dimensiones: (551, 10)


### 2.3 Selección de columnas estándar

Se conservan únicamente las columnas presentes en todas las temporadas:
las necesarias para el **matching** con Sofascore y las **variables salariales** objetivo.

In [32]:
# Columnas para el matching con Sofascore
COLS_MATCHING = ['player', 'club', 'position', 'age', 'nationality', 'data_country', 'data_season']

# Variables salariales objetivo
COLS_SALARY = ['gross_weekly_eur', 'gross_annual_eur']

COLS_FINAL = COLS_MATCHING + COLS_SALARY

df_proto = df_proto[COLS_FINAL]

print(f'✅ Columnas seleccionadas: {COLS_FINAL}')
df_proto.head(5)

✅ Columnas seleccionadas: ['player', 'club', 'position', 'age', 'nationality', 'data_country', 'data_season', 'gross_weekly_eur', 'gross_annual_eur']


,player,club,position,age,nationality,data_country,data_season,gross_weekly_eur,gross_annual_eur
0,Robert Lewandowski,Barcelona,F,36,Poland,spain,2425.0,"€ 640,962","€ 33,330,000"
1,Kylian Mbappé,Real Madrid,F,26,France,spain,2425.0,"€ 600,962","€ 31,250,000"
2,Vinicius Junior,Real Madrid,F,24,Brazil,spain,2425.0,"€ 480,769","€ 25,000,000"
3,David Alaba,Real Madrid,D,32,Austria,spain,2425.0,"€ 432,692","€ 22,500,000"
4,Jan Oblak,Atletico Madrid,K,32,Slovenia,spain,2425.0,"€ 400,577","€ 20,830,000"


### 2.4 Parseo de valores monetarios

Los salarios vienen como strings con formato europeo (`"€ 33,330,000"`).
Se eliminan el símbolo `€`, los espacios y las comas, y se convierten a `float`.

In [33]:
def parsear_euros(serie: pd.Series) -> pd.Series:
    """Convierte una serie de strings tipo '€ 33,330,000' a float."""
    return (
        serie
        .str.replace('€', '', regex=False)
        .str.replace(',', '', regex=False)
        .str.strip()
        .astype(float)
    )

df_proto['gross_weekly_eur'] = parsear_euros(df_proto['gross_weekly_eur'])
df_proto['gross_annual_eur'] = parsear_euros(df_proto['gross_annual_eur'])

print('✅ Valores monetarios convertidos a float')
print()
print(df_proto[['player', 'gross_weekly_eur', 'gross_annual_eur']].head(3).to_string(index=False))

✅ Valores monetarios convertidos a float

            player  gross_weekly_eur  gross_annual_eur
Robert Lewandowski          640962.0        33330000.0
     Kylian Mbappé          600962.0        31250000.0
   Vinicius Junior          480769.0        25000000.0


### 2.5 Corrección de tipos de datos

In [34]:
df_proto['age']         = df_proto['age'].astype(int)
df_proto['data_season'] = df_proto['data_season'].astype(float).astype(int).astype(str)

print('✅ Tipos corregidos')
print(df_proto.dtypes)

✅ Tipos corregidos
player               object
club                 object
position             object
age                   int64
nationality          object
data_country         object
data_season          object
gross_weekly_eur    float64
gross_annual_eur    float64
dtype: object


### 2.6 Validación del prototipo

In [35]:
assert list(df_proto.columns) == COLS_FINAL,         'ERROR: el orden de columnas no es el esperado'
assert df_proto['gross_annual_eur'].dtype == float,  'ERROR: gross_annual_eur no es float'
assert df_proto['age'].dtype == int,                 'ERROR: age no es int'
assert df_proto['data_season'].dtype == object,      'ERROR: data_season no es str'
assert df_proto.isnull().sum().sum() == 0,           'ERROR: hay valores nulos inesperados'

print('✅ Todas las validaciones superadas')
print(f'   Dimensiones finales: {df_proto.shape}')
df_proto.head(5)

✅ Todas las validaciones superadas
   Dimensiones finales: (551, 9)


,player,club,position,age,nationality,data_country,data_season,gross_weekly_eur,gross_annual_eur
0,Robert Lewandowski,Barcelona,F,36,Poland,spain,2425,640962.0,33330000.0
1,Kylian Mbappé,Real Madrid,F,26,France,spain,2425,600962.0,31250000.0
2,Vinicius Junior,Real Madrid,F,24,Brazil,spain,2425,480769.0,25000000.0
3,David Alaba,Real Madrid,D,32,Austria,spain,2425,432692.0,22500000.0
4,Jan Oblak,Atletico Madrid,K,32,Slovenia,spain,2425,400577.0,20830000.0


---
## 3. Función generalizada `limpiar_capology()`

Se encapsula toda la lógica anterior en una función reutilizable.
La función es robusta ante las columnas extra presentes en algunos años (ej. 25/26),
seleccionando siempre únicamente el subconjunto estándar.

In [36]:
COLS_MATCHING = ['player', 'club', 'position', 'age', 'nationality', 'data_country', 'data_season']
COLS_SALARY   = ['gross_weekly_eur', 'gross_annual_eur']
COLS_FINAL    = COLS_MATCHING + COLS_SALARY


def parsear_euros(serie: pd.Series) -> pd.Series:
    """Convierte una serie de strings tipo '€ 33,330,000' a float."""
    return (
        serie
        .str.replace('€', '', regex=False)
        .str.replace(',', '', regex=False)
        .str.strip()
        .astype(float)
    )


def limpiar_capology(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1. Extraer el header real de la fila 0 y renombrar columnas dinámicamente.
    # Se usa 'not in rename_dinamico.values()' para evitar asignar el mismo nombre
    # a dos columnas cuando el archivo tiene columnas extra (ej. 25/26).
    header_real = df.iloc[0]
    rename_dinamico = {}
    for col, valor in header_real.items():
        if isinstance(valor, str):
            v = valor.strip().replace('\n', ' ')
            if 'PLAYER'      in v and 'player'           not in rename_dinamico.values():
                rename_dinamico[col] = 'player'
            elif 'GROSS P/W' in v and 'gross_weekly_eur' not in rename_dinamico.values():
                rename_dinamico[col] = 'gross_weekly_eur'
            elif 'GROSS P/Y' in v and 'gross_annual_eur' not in rename_dinamico.values():
                rename_dinamico[col] = 'gross_annual_eur'
            elif 'POS.'      in v and 'position'         not in rename_dinamico.values():
                rename_dinamico[col] = 'position'
            elif v == 'AGE'  and 'age'                   not in rename_dinamico.values():
                rename_dinamico[col] = 'age'
            elif 'COUNTRY'   in v and 'nationality'      not in rename_dinamico.values():
                rename_dinamico[col] = 'nationality'
            elif v == 'CLUB' and 'club'                  not in rename_dinamico.values():
                rename_dinamico[col] = 'club'

    df = df.rename(columns=rename_dinamico)

    # 2. Eliminar fila 0
    df = df.iloc[1:].reset_index(drop=True)

    # 3. Seleccionar columnas estándar
    df = df[COLS_FINAL]

    # 4. Parsear valores monetarios
    df['gross_weekly_eur'] = parsear_euros(df['gross_weekly_eur'])
    df['gross_annual_eur'] = parsear_euros(df['gross_annual_eur'])

    # 5. Corregir tipos
    df['age']         = df['age'].astype(int)
    df['data_season'] = df['data_season'].astype(float).astype(int).astype(str)

    return df


print('✅ Función limpiar_capology() definida')

✅ Función limpiar_capology() definida


---
## 4. Aplicación sobre los 36 archivos

Se procesan todos los CSVs de `data/raw/capology/` y se guardan los resultados
en `data/processed/capology/` manteniendo el mismo nombre de archivo.

In [37]:
archivos_raw = sorted(RAW_DIR.glob('*.csv'))
print(f'Archivos encontrados: {len(archivos_raw)}')
print()

resumen = []

for archivo in archivos_raw:
    try:
        df_raw   = pd.read_csv(archivo)
        df_clean = limpiar_capology(df_raw)

        df_clean.to_csv(PROCESSED_DIR / archivo.name, index=False)

        resumen.append({
            'archivo' : archivo.name,
            'filas'   : len(df_clean),
            'columnas': df_clean.shape[1],
            'estado'  : '✅'
        })

    except Exception as e:
        resumen.append({
            'archivo' : archivo.name,
            'filas'   : None,
            'columnas': None,
            'estado'  : f'❌ {e}'
        })

df_resumen = pd.DataFrame(resumen)
print(df_resumen.to_string(index=False))
print()
print(f'Procesados correctamente: {(df_resumen["estado"] == "✅").sum()} / {len(df_resumen)}')

Archivos encontrados: 36



            archivo  filas  columnas estado
cg_england_2021.csv    593         9      ✅
cg_england_2122.csv    562         9      ✅
cg_england_2223.csv    593         9      ✅
cg_england_2324.csv    610         9      ✅
cg_england_2425.csv    770         9      ✅
cg_england_2526.csv    706         9      ✅
 cg_france_2021.csv    586         9      ✅
 cg_france_2122.csv    631         9      ✅
 cg_france_2223.csv    629         9      ✅
 cg_france_2324.csv    550         9      ✅
 cg_france_2425.csv    561         9      ✅
 cg_france_2526.csv    501         9      ✅
cg_germany_2021.csv    552         9      ✅
cg_germany_2122.csv    569         9      ✅
cg_germany_2223.csv    571         9      ✅
cg_germany_2324.csv    557         9      ✅
cg_germany_2425.csv    566         9      ✅
cg_germany_2526.csv    556         9      ✅
  cg_italy_2021.csv    652         9      ✅
  cg_italy_2122.csv    660         9      ✅
  cg_italy_2223.csv    649         9      ✅
  cg_italy_2324.csv    659      